# 02 · Routing dynamics — the confidence-band state machine

The SAME document, classified at five different mock confidence levels, takes five different paths through the graph.

## Setup — the lab bench

In [1]:
import json
import sys
from pathlib import Path

# Work from the repo root no matter where the kernel was started.
ROOT = Path.cwd()
while ROOT != ROOT.parent and not (ROOT / "pyproject.toml").exists():
    ROOT = ROOT.parent
assert (ROOT / "notebooks" / "pipeline_lab.py").exists(), (
    f"llm-mailroom repo root not found above {Path.cwd()}"
)
sys.path.insert(0, str(ROOT / "notebooks"))

import pipeline_lab as lab


## What you'll see

- the live threshold table (read from `taxonomy.yaml`, never copy-pasted)
- a confidence matrix: 0.98 / 0.80 / 0.60 / 0.30 through the same document
- the medium band's ONE re-classification pass, then Lane A
- where every threshold literal lives

**Honesty label:** every run below is the REAL pipeline (`graph.build_graph.run_pipeline`) against the mock LLM seam from `src/tests/conftest.py` — mocked intelligence, real machinery.

## The bands (live from the taxonomy)

In [2]:
import json
lab.band_report()

{'high': 0.95, 'low': 0.7, 'judge_band_high': 0.85}

## One document, five confidences

In [3]:
results = {}
with lab.lab_sandbox() as env:
    for c in (0.98, 0.80, 0.60, 0.30):
        r = lab.run_document(
            env,
            lab.DOC_CONTRACT,
            classification={**lab.CLASSIFY_CONTRACT_HIGH, "confidence": c},
            extraction=lab.EXTRACT_HIGH,
            filename=f"matrix_{int(c * 100)}.txt",
        )
        results[c] = {
            "path": lab.path_of(r["steps"]),
            "stage": r["final"]["stage"],
            "classification_attempts": r["final"]["classification_attempts"],
        }

for c, info in results.items():
    print(f"confidence {c:>4}: stage={info['stage']:<8} "
          f"attempts={info['classification_attempts']}")
    print(f"              {' -> '.join(info['path'])}")


confidence 0.98: stage=archived attempts=1
              ingest-document -> classify-document -> extract-fields -> compile-report -> write-catalog -> archive-document
confidence  0.8: stage=review   attempts=2
              ingest-document -> classify-document -> route-for-review
confidence  0.6: stage=review   attempts=2
              ingest-document -> classify-document -> route-for-review
confidence  0.3: stage=review   attempts=2
              ingest-document -> classify-document -> route-for-review


## Reading the machine

- **>= 0.95 (high):** straight through classify -> extract. No detour.
- **0.70 - 0.95 (medium):** ONE re-classification pass first (the retry
  prompt often resolves ambiguity), then the agent second opinion (Lane A).
- **< 0.70 (low):** also gets its one retry — but after it, the graph sends
  the document to the human review siding rather than the agent lane.

Note `classification_attempts` reaches 2 on EVERY banded path: the retry
budget is spent identically; what differs is *who adjudicates* the survivor
(Lane A reviewer vs human).

## Medium band, survived: the second opinion wins

The medium document gets a reviewer pass. When the reviewer answers with >= 0.95 confidence, ITS label wins and the original specialist handoff changes accordingly (here: contract -> court opinion):

In [4]:
with lab.lab_sandbox() as env:
    # Reviewer overrides the sorter's 'contract' with 'court_opinion' @ 0.97.
    lab.script_client(
        env["client"],
        reviewer=lab.REVIEWER_OVERRIDE,
        specialist={"court opinion": lab.COURT_OPINION_EXTRACTION},
    )
    r = lab.run_document(
        env,
        lab.DOC_CONTRACT,
        classification=lab.CLASSIFY_CONTRACT_MEDIUM,
        extraction=lab.COURT_OPINION_EXTRACTION,
        filename="laneA_override.txt",
    )
    f = r["final"]
    print("path:      ", " -> ".join(lab.path_of(r["steps"])))
    print("verdict:   ", f["review_verdict"])
    print("final type:", f["doc_type"], "| stage:", f["stage"])


path:       ingest-document -> classify-document -> extract-fields -> compile-report -> write-catalog -> archive-document
verdict:    reviewer_overrides
final type: court_opinion | stage: archived


## Where the numbers live

Every threshold above is read at RUNTIME from `config/taxonomy.yaml`
(`pipeline.confidence_thresholds`) — `band_report()` just renders them.
Change the YAML, rerun this notebook, and the paths move with it. The
routers themselves are `graph/routing.py::after_classify`,
`after_retry_classify`, and `after_review_classify`.

## Where to go next

- **03 · review_lanes** — inside Lane A and Lane B in detail
- **04 · human_in_the_loop** — the review siding this notebook kept landing in